In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Filter eligible part keys
target_keys = part.loc[
    (part.P_BRAND == "Brand#23") & (part.P_CONTAINER == "MED BOX"),
    "P_PARTKEY"
]

In [ ]:
### cell 1 ###

# Subset lineitem to only those keys and needed cols
li = lineitem.loc[
    lineitem.L_PARTKEY.isin(target_keys),
    ["L_PARTKEY", "L_QUANTITY", "L_EXTENDEDPRICE"]
]

# Compute per-part threshold and filter rows
good = li[ li.L_QUANTITY <
    li.groupby("L_PARTKEY", sort=False)["L_QUANTITY"].transform("mean") * 0.2
]

In [ ]:
### cell 2 ###

# Sum and scale
total = pd.DataFrame({
    "avg_yearly": [ good.L_EXTENDEDPRICE.sum() / 7.0 ]
})